In [1]:
import sys, os, tempfile, timeit, pickle, inspect
from dsc.dsc_io import load_dsc as __load_dsc__, source_dirs as __source_dirs__
import numpy as np

In [2]:
simres = __load_dsc__(['/gpfs/commons/groups/knowles_lab/sbanerjee/low_rank_matrix_approximation_numerical_experiments/lrma/blockdiag_p/blockdiag_p_1.pkl','/gpfs/commons/groups/knowles_lab/sbanerjee/low_rank_matrix_approximation_numerical_experiments/lrma/flashier/blockdiag_p_1_identical_1_flashier_1.rds'])

INFO: Note: NumExpr detected 56 cores but "NUMEXPR_MAX_THREADS" not set, so enforcing safe limit of 8.
INFO: NumExpr defaulting to 8 threads.
INFO: cffi mode is CFFI_MODE.ANY
INFO: R home found: /nfs/sw/easybuild/software/R/4.4.1-gfbf-2023b/lib64/R
INFO: R library path: /nfs/sw/easybuild/software/R/4.4.1-gfbf-2023b/lib64/R/lib:/nfs/sw/easybuild/software/FriBidi/1.0.13-GCCcore-13.2.0/lib64:/nfs/sw/easybuild/software/FriBidi/1.0.13-GCCcore-13.2.0/lib:/nfs/sw/easybuild/software/HarfBuzz/8.2.2-GCCcore-13.2.0/lib64:/nfs/sw/easybuild/software/HarfBuzz/8.2.2-GCCcore-13.2.0/lib:/nfs/sw/easybuild/software/Tk/8.6.13-GCCcore-13.2.0/lib64:/nfs/sw/easybuild/software/Tk/8.6.13-GCCcore-13.2.0/lib:/nfs/sw/easybuild/software/cURL/8.3.0-GCCcore-13.2.0/lib64:/nfs/sw/easybuild/software/cURL/8.3.0-GCCcore-13.2.0/lib:/nfs/sw/easybuild/software/OpenSSL/1.1/lib64:/nfs/sw/easybuild/software/OpenSSL/1.1/lib:/nfs/sw/easybuild/software/libgit2/1.7.2-GCCcore-13.2.0/lib64:/nfs/sw/easybuild/software/libgit2/1.7.2-GC

In [3]:
simres.keys()

dict_keys(['Z', 'Zmask', 'effect_size_obs', 'effect_size_true', 'Ltrue', 'Ftrue', 'Mtrue', 'Ctrue', 'nsample', 'DSC_DEBUG', 'L_est', 'F_est', 'S2'])

In [19]:
F = simres['F_est']
Ftrue = simres['Ftrue']
L = simres['L_est']
Ltrue = simres['Ltrue']
labels = simres['Ctrue']
Ztrue = simres['Z']

In [52]:
import numpy as np
from scipy.spatial import procrustes
from sklearn.cluster import AgglomerativeClustering
from sklearn import metrics as skmetrics


def standardize(X, axis = 0, center = True, scale = True):
    if center:
        X = X - np.mean(X, axis = axis, keepdims = True)
    if scale:
        X /= np.std(X, axis = axis, keepdims = True)
    return X


def mean_squared_error(original, recovered, mask = None):
    if mask is None: mask = np.ones_like(original)
    n = np.sum(mask)
    mse = np.sum(np.square((original - recovered) * mask)) / n
    return mse


def root_mean_squared_error(original, recovered, mask = None):
    mse = mean_squared_error(original, recovered, mask = mask)
    return np.sqrt(mse)


def peak_signal_to_noise_ratio(original, recovered, mask = None):
    if mask is None: mask = np.ones_like(original)
    omax = np.max(original[mask == 1])
    omin = np.min(original[mask == 1])
    maxsig2 = np.square(omax - omin)
    mse = mean_squared_error(original, recovered, mask)
    res = 10 * np.log10(maxsig2 / mse)
    return res


def matrix_dissimilarity_scores(original, recovered, mask = None, match = 'zerofill'):
    '''
    Procrustes analysis returns the square of the Frobenius norm.
    Use the rotated matrix to obtain the peak signal-to-noise ratio (PSNR).
    Input matrices can have different dimensions.
    There are two ways to match:
        - clip: remove information from the larger matrix
        - zerofill: pad zero columns in the smaller matrix
    '''
    n_orig = original.shape[1]
    n_recv = recovered.shape[1]
    m = original.shape[0]
    if match == 'clip':
        n = min(n_orig, n_recv)
        X = original[:, :n]
        Y = recovered[:, :n]
    elif match == 'zerofill':
        n = max(n_orig, n_recv)
        X = np.zeros((m, n))
        Y = np.zeros((m, n))
        X[:, :n_orig] = original
        Y[:, :n_recv] = recovered
    # gleanr sometimes produces a single column of zero values
    # procrustes requires: Input matrices must contain >1 unique points
    # a matrix has no unique points iff max - min == 0.
    if np.ptp(Y) == 0 :
        m2 = np.sum(np.square(X))
        psnr = peak_signal_to_noise_ratio(X, Y, mask)
    else:
        R_orig, R_recv, m2 = procrustes(X, Y)
        psnr = peak_signal_to_noise_ratio(R_orig, R_recv, mask)
    # procrustes produces squared error, not the mean error
    ndim = m * n
    rmse = np.sqrt(m2 / ndim)
    return rmse, psnr


def adjusted_mutual_information_score(X, class_labels):
    X = standardize(X, axis = 0, center = True, scale = False)
    # we know the true clusters
    n_clusters = len(set(class_labels))
    distance_matrix = skmetrics.pairwise.pairwise_distances(X, metric='euclidean')
    model_exact = AgglomerativeClustering(n_clusters = n_clusters, linkage = 'average', metric = 'precomputed')
    class_pred_exact = model_exact.fit_predict(distance_matrix)
    mi_score = skmetrics.mutual_info_score(class_labels, class_pred_exact)
    model_approx = AgglomerativeClustering(n_clusters = n_clusters + 2, linkage = 'average', metric = 'precomputed')
    class_pred_approx = model_approx.fit_predict(distance_matrix)
    adj_mi_score = skmetrics.adjusted_mutual_info_score(class_labels, class_pred_approx)
    return mi_score, adj_mi_score

In [53]:
Ztrue = standardize(Ltrue @ Ftrue.T, axis = 0)
Zrecv = standardize(L @ F.T, axis = 0)
Z_rmse = root_mean_squared_error(Ztrue, Zrecv)
MI, adj_MI = adjusted_mutual_information_score(L, labels)
print(Z_rmse, MI, adj_MI)

0.37260870918877204 0.015960386519815056 0.006762637288238737


In [49]:
L_cent = standardize(L, axis = 0, center = True, scale = False)

In [51]:
np.mean(L_cent, axis = 1)

array([-0.18308754, -0.00451633, -0.27708105, -0.11610001, -0.15283315,
       -0.51274284, -0.11355322, -0.28200384, -0.66439401, -0.29811958,
       -0.37769487, -0.54758163, -0.07178633,  0.08149472,  0.30396093,
       -0.08925525, -0.16898508, -0.71117415,  0.01513208, -0.2644866 ,
       -0.18969595,  0.15380002,  0.10216525, -0.16998496, -0.41334468,
       -0.15109791,  0.00514893,  0.46383046,  0.1313887 , -0.04828475,
       -0.35568481, -0.71937276, -0.41695068, -0.80022997, -0.41676768,
       -0.38693789, -0.18127968,  0.10799638, -0.36507386, -0.09461297,
       -0.2084387 , -0.55202061, -0.43712338, -0.49098293, -0.24320354,
        0.09561604,  0.02724635, -0.24808649, -0.561515  , -0.42648247,
       -0.3431085 , -0.22378978, -0.52549177,  0.05491525, -0.33985926,
        0.13077745, -0.00217471, -0.19141082, -0.01840268,  0.04573848,
       -0.38588973,  0.00272839, -0.26775315, -0.12038331, -0.15241513,
        0.08014139, -0.06744927, -0.14539008,  0.20577637,  0.05

In [ ]:
L_rmse, L_psnr = matrix_dissimilarity_scores(Ltrue, L)
F_rmse, F_psnr = matrix_dissimilarity_scores(Ftrue, F)
Ztrue = Ltrue @ Ftrue.T
Zrecv = L @ F.T
Z_rmse = root_mean_squared_error(Ztrue, Zrecv, center = True, scale = True)
Z_psnr = peak_signal_to_noise_ratio(Ztrue, Zrecv, center = True, scale = True)
adj_MI = adjusted_mutual_information_score(L, labels)

In [49]:
np.std(Ztrue, axis = 0)

array([0.01197236, 0.01021022, 0.01541143, 0.0167161 , 0.01076241,
       0.01090559, 0.01253327, 0.01068817, 0.01594445, 0.01168677,
       0.01753007, 0.01084627, 0.01251287, 0.01431051, 0.01451388,
       0.01535136, 0.00570639, 0.01761673, 0.01299931, 0.01109974,
       0.01217678, 0.01616634, 0.01438111, 0.00970669, 0.01760455,
       0.01222692, 0.01096018, 0.0116218 , 0.01910758, 0.01076785,
       0.0103603 , 0.00818124, 0.00911218, 0.01176709, 0.01124645,
       0.01583916, 0.01569032, 0.01475234, 0.01832566, 0.01416755,
       0.01067234, 0.01038901, 0.01108383, 0.01106339, 0.01400645,
       0.0196056 , 0.01497416, 0.01028227, 0.01013497, 0.01258975,
       0.01909988, 0.01624988, 0.02075946, 0.01435338, 0.01007985,
       0.00966959, 0.02073014, 0.01402894, 0.01658447, 0.0124186 ,
       0.01243057, 0.01251298, 0.00790897, 0.01341745, 0.00986987,
       0.01226954, 0.00982666, 0.01514851, 0.01786859, 0.01674185,
       0.01486151, 0.01305984, 0.00924357, 0.01403529, 0.00774

In [42]:
matrix_dissimilarity_scores(Ztrue, Zrecv)

(0.0008339235404871867, 32.39916376119471)

In [43]:
Z_orig, Z_recv, m2 = procrustes(Ztrue, Zrecv)

In [44]:
np.sqrt(mean_squared_error(Z_orig, Z_recv))

0.0008339235404871867

In [25]:
np.sqrt(np.sum(np.square(Ztrue))/np.prod(Ztrue.shape))

3.2175714066723415

In [26]:
np.sqrt(np.sum(np.square(Z_orig))/np.prod(Ztrue.shape))

0.0031622776601683794

In [32]:
np.std(Z_orig, axis = 0)

array([0.00285101, 0.00285032, 0.00303456, 0.00346569, 0.00287549,
       0.00264096, 0.00275695, 0.00302902, 0.00349391, 0.0031252 ,
       0.00391794, 0.00285848, 0.00323073, 0.00309226, 0.00331745,
       0.00321569, 0.00230729, 0.0033862 , 0.00334837, 0.00285328,
       0.00331059, 0.00395226, 0.00292655, 0.00272252, 0.00329845,
       0.00294404, 0.00303934, 0.0028296 , 0.00370438, 0.00302924,
       0.00306805, 0.00286639, 0.00279878, 0.00280728, 0.00295224,
       0.00315495, 0.00349494, 0.00305681, 0.00339553, 0.00339924,
       0.00296825, 0.00322038, 0.00292824, 0.00293096, 0.00333743,
       0.00404822, 0.00349667, 0.0028304 , 0.00266007, 0.00295629,
       0.00352913, 0.00368568, 0.00410794, 0.00342385, 0.00248591,
       0.00276599, 0.00421211, 0.00323803, 0.00341518, 0.00316269,
       0.00320236, 0.00324744, 0.00228503, 0.00306566, 0.00254952,
       0.00280856, 0.00300441, 0.00323556, 0.00363113, 0.00346127,
       0.00315485, 0.00294249, 0.00319821, 0.00339492, 0.00270

In [28]:
Ztrue

array([[ 3.80216886,  2.6088751 , -3.13107357, ..., -2.73098774,
        -1.72971793, -0.53958383],
       [ 1.75865909,  0.64646204,  0.70644361, ...,  1.86258995,
         0.14739632,  0.11174886],
       [ 2.91529335,  3.00576069, -0.3339158 , ...,  1.25565775,
        -1.01700378,  2.41667857],
       ...,
       [-3.38385116, -2.82521915, -0.56789255, ..., -4.51749305,
         5.96669918,  0.89889546],
       [ 2.48834388,  2.54316007,  3.62053463, ..., -5.32958302,
         3.95956721, -3.4522214 ],
       [ 1.21412068,  1.73593163,  1.29908731, ..., -1.4484917 ,
         2.67803099, -3.32434321]])